# Cleaning Dataset Tanaman Pangan — Kabupaten Kediri

Membersihkan dua file (tiap file punya struktur berbeda, jadi dibersihkan terpisah):

| File | Isi | Struktur asli |
|---|---|---|
| `Tanaman_Pangan.csv` | luas panen, produksi, produktivitas per komoditas | Tingkat **Kabupaten**, 5 baris. Angka dan satuan menyatu dalam satu sel (`"34198.23 Ha"`) |
| `Luas_Lahan_Per_Kecamatan.csv` | penggunaan lahan per kecamatan (BPS) | 4 baris judul bertingkat (bahasa Inggris), lalu 26 kecamatan dengan format `[010] Mojo`. Tahun ada di baris judul |

Notebook ini **hanya membersihkan**. Tidak ada pembagian/estimasi ke kecamatan di sini.

**Output** (`data/clean/tanaman_pangan/`):
- `tanaman_pangan_kabupaten.csv` → `komoditas, luas_panen_ha, produksi_ton, produktivitas_kuintal_ha`
- `luas_lahan_per_kecamatan.csv` → `kode_kecamatan, kecamatan, tahun, luas_sawah_ha, luas_pertanian_bukan_sawah_ha, luas_bukan_pertanian_ha, luas_total_ha`

Kolom `tahun` dan `kode_kecamatan` di file lahan memang ada di sumbernya (bukan metadata tambahan).

In [1]:
# 1. IMPORT LIBRARY
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

✓ Library siap.


In [2]:
# 2. CARI LOKASI PROJECT (folder 'data') SECARA OTOMATIS
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan dari direktori notebook ini ke atas. "
        "Pastikan notebook dijalankan dari dalam repository (mis. dari folder 'notebooks/')."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
OUT_DIR = os.path.join(DATA_DIR, "clean", "tanaman_pangan")
os.makedirs(OUT_DIR, exist_ok=True)

# File lahan ikut disimpan di folder yang sama sesuai permintaan. Kalau mau dipisah,
# ganti ke os.path.join(DATA_DIR, "clean", "lahan") lalu buat foldernya.
OUT_DIR_LAHAN = OUT_DIR

print("Project root:", PROJECT_ROOT)
print("Output      :", OUT_DIR)

Project root: d:\SDT\Project Analitika Data\AgriDash-0.1
Output      : d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\tanaman_pangan


In [3]:
# 3. CARI FILE SUMBER
# Dicocokkan longgar lewat nama (mis. 'Tanaman_Pangan.csv' atau 'tanaman_pangan_kabupaten.csv').
# File turunan yang namanya memuat 'kecamatan' sengaja dikecualikan dari pencarian tanaman pangan.

def cari_file(wajib_ada, dilarang=(), folder_utama=None):
    kandidat = []
    for root, _, files in os.walk(RAW_DIR):
        for f in files:
            nama = f.lower()
            if nama.endswith(".csv") and all(k in nama for k in wajib_ada) \
                    and not any(k in nama for k in dilarang):
                kandidat.append(os.path.join(root, f))
    kandidat.sort(key=lambda p: (os.path.basename(os.path.dirname(p)) != folder_utama, p))
    if not kandidat:
        raise FileNotFoundError(f"File dengan kata kunci {wajib_ada} tidak ditemukan di {RAW_DIR}")
    if len(kandidat) > 1:
        print(f"⚠️  Lebih dari satu kandidat untuk {wajib_ada}, dipakai yang pertama:")
        for k in kandidat:
            print("    ", k)
    return kandidat[0]

FILE_PANGAN = cari_file(["tanaman", "pangan"], dilarang=["kecamatan"], folder_utama="tanaman_pangan")
FILE_LAHAN = cari_file(["luas", "lahan"], folder_utama="lahan")

print("Tanaman pangan:", FILE_PANGAN)
print("Luas lahan    :", FILE_LAHAN)

Tanaman pangan: d:\SDT\Project Analitika Data\AgriDash-0.1\data\raw\tanaman_pangan\Tanaman Pangan.csv
Luas lahan    : d:\SDT\Project Analitika Data\AgriDash-0.1\data\raw\tanaman_pangan\Luas Lahan Per Kecamatan.csv


In [4]:
# 4. MASTER KECAMATAN & FUNGSI BANTU

MASTER_KECAMATAN = [
    "Mojo", "Semen", "Ngadiluwih", "Kras", "Ringinrejo", "Kandat", "Wates", "Ngancar",
    "Plosoklaten", "Gurah", "Puncu", "Kepung", "Kandangan", "Pare", "Badas", "Kunjang",
    "Plemahan", "Purwoasri", "Papar", "Pagu", "Kayenkidul", "Gampengrejo", "Ngasem",
    "Banyakan", "Grogol", "Tarokan",
]
KOREKSI_KECAMATAN = {"Kayen Kidul": "Kayenkidul"}


def ke_angka(teks):
    # '-' -> 0.0 (konvensi BPS: tidak ada). '...' / kosong -> NaN (tidak tersedia).
    # Mengenali format Indonesia (1.234,56) dan internasional (1,234.56).
    if pd.isna(teks):
        return np.nan
    s = str(teks).strip()
    if s == "-":
        return 0.0
    if s in ("", "...", "nan"):
        return np.nan
    s = re.sub(r"[^0-9.,-]", "", s)
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        if re.fullmatch(r"-?[1-9]\d{0,2}(,\d{3})+", s):
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


def rapikan_nama(teks):
    return re.sub(r"\s+", " ", str(teks)).strip()


print("✓ Master kecamatan & fungsi bantu siap.")

✓ Master kecamatan & fungsi bantu siap.


In [5]:
# 5. TANAMAN PANGAN (KABUPATEN) — BACA

# dtype=str supaya angka+satuan ('34198.23 Ha') tidak diubah pandas.
df_raw_pangan = pd.read_csv(FILE_PANGAN, encoding="utf-8-sig", dtype=str, keep_default_na=False)

kolom_wajib = ["Tanaman", "Luas Panen", "Produksi", "Produktivitas"]
kurang = [k for k in kolom_wajib if k not in df_raw_pangan.columns]
if kurang:
    raise ValueError(f"Kolom tidak ditemukan: {kurang}. Kolom yang ada: {list(df_raw_pangan.columns)}")

# Buang baris yang nama tanamannya kosong (mis. baris kosong di akhir file).
df_raw_pangan = df_raw_pangan[df_raw_pangan["Tanaman"].str.strip() != ""].reset_index(drop=True)
print(f"{len(df_raw_pangan)} baris komoditas terbaca.")
display(df_raw_pangan)

5 baris komoditas terbaca.


,No,Tanaman,Luas Panen,Produksi,Produktivitas
0,1,Padi,34198.23 Ha,198324.32 Ton,57.992568621242 Kuintal/Ha
1,2,Kedelai,1.9 Ha,2.5030348615385 Ton,13.17386769 Kuintal/Ha
2,3,Kacang Tanah,476.5 Ha,843.69645825108 Ton,17.706116647452 Kuintal/Ha
3,4,Ubi Jalar,538.5 Ha,15825.8317 Ton,293.88731104921 Kuintal/Ha
4,5,Ketela Pohon,652.8 Ha,18597.024533333 Ton,284.88089052287 Kuintal/Ha


In [6]:
# 6. TANAMAN PANGAN — BERSIHKAN (pisahkan angka dan satuan)

POLA_ANGKA_SATUAN = re.compile(r"^\s*(-|\.\.\.|[0-9][0-9.,]*)\s*(.*?)\s*$")

def pisah_angka_satuan(teks):
    m = POLA_ANGKA_SATUAN.match(str(teks))
    if not m:
        return np.nan, None
    return ke_angka(m.group(1)), m.group(2).strip().lower()

# Nama kolom output memuat satuannya, jadi satuan di file HARUS sesuai. Kalau tidak, berhenti
# (lebih baik error daripada diam-diam salah label).
SATUAN_DIHARAPKAN = {"Luas Panen": "ha", "Produksi": "ton", "Produktivitas": "kuintal/ha"}
NAMA_KOLOM_OUTPUT = {"Luas Panen": "luas_panen_ha", "Produksi": "produksi_ton",
                     "Produktivitas": "produktivitas_kuintal_ha"}

hasil = {"komoditas": df_raw_pangan["Tanaman"].apply(rapikan_nama)}
for kolom, satuan_benar in SATUAN_DIHARAPKAN.items():
    pasangan = df_raw_pangan[kolom].apply(pisah_angka_satuan)
    angka = pasangan.apply(lambda p: p[0])
    satuan_ditemukan = set(pasangan.apply(lambda p: p[1]))
    if satuan_ditemukan != {satuan_benar}:
        raise ValueError(
            f"Satuan kolom '{kolom}' = {satuan_ditemukan}, diharapkan {{'{satuan_benar}'}}. "
            "Cek file sumber sebelum lanjut."
        )
    if angka.isna().any():
        baris_gagal = df_raw_pangan.loc[angka.isna(), ["Tanaman", kolom]]
        print(f"⚠️  Ada nilai '{kolom}' yang tidak terbaca:")
        display(baris_gagal)
    hasil[NAMA_KOLOM_OUTPUT[kolom]] = angka

df_pangan = pd.DataFrame(hasil)
display(df_pangan)

,komoditas,luas_panen_ha,produksi_ton,produktivitas_kuintal_ha
0,Padi,34198.23,198324.320000,57.992569
1,Kedelai,1.90,2.503035,13.173868
2,Kacang Tanah,476.50,843.696458,17.706117
3,Ubi Jalar,538.50,15825.831700,293.887311
4,Ketela Pohon,652.80,18597.024533,284.880891


In [7]:
# 7. TANAMAN PANGAN — VALIDASI

assert not df_pangan["komoditas"].duplicated().any(), "Ada komoditas ganda."
assert (df_pangan[["luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]] >= 0).all().all()

# a) Produktivitas di file harus sama dengan produksi / luas panen (ton/ha x 10 = kuintal/ha).
hitung = df_pangan["produksi_ton"] / df_pangan["luas_panen_ha"] * 10
selisih = (hitung - df_pangan["produktivitas_kuintal_ha"]).abs() / hitung
tidak_cocok = df_pangan.loc[selisih > 0.01, "komoditas"].tolist()
if tidak_cocok:
    print(f"⚠️  Produktivitas tidak konsisten dengan produksi/luas panen (>1%): {tidak_cocok}")
else:
    print("✓ Produktivitas konsisten dengan produksi / luas panen untuk semua komoditas.")

# b) Komoditas di proposal yang belum ada di data (Ketela Pohon = Ubi Kayu).
SINONIM = {"Ketela Pohon": "Ubi Kayu"}
KOMODITAS_PROPOSAL = ["Padi", "Jagung", "Kedelai", "Kacang Tanah", "Kacang Hijau", "Ubi Kayu", "Ubi Jalar"]
ada = {SINONIM.get(k, k) for k in df_pangan["komoditas"]}
belum_ada = [k for k in KOMODITAS_PROPOSAL if k not in ada]
if belum_ada:
    print(f"⚠️  Komoditas di proposal yang belum ada di file ini: {belum_ada}")

# c) Luas panen sangat kecil untuk satu kabupaten (kemungkinan salah baris/satuan/tahun).
BATAS_LUAS_KECIL_HA = 10
kecil = df_pangan[df_pangan["luas_panen_ha"] < BATAS_LUAS_KECIL_HA]
for _, r in kecil.iterrows():
    print(f"⚠️  {r['komoditas']}: luas panen kabupaten hanya {r['luas_panen_ha']} ha. Cek ke sumber BPS.")

print("⚠️  File ini tidak punya kolom tahun. Catat tahun datanya dari sumber.")

✓ Produktivitas konsisten dengan produksi / luas panen untuk semua komoditas.
⚠️  Komoditas di proposal yang belum ada di file ini: ['Jagung', 'Kacang Hijau']
⚠️  Kedelai: luas panen kabupaten hanya 1.9 ha. Cek ke sumber BPS.
⚠️  File ini tidak punya kolom tahun. Catat tahun datanya dari sumber.


In [8]:
# 8. LUAS LAHAN PER KECAMATAN — BACA & KENALI STRUKTUR

# header=None + skip_blank_lines=False: baris judul dibaca apa adanya, lalu dikenali isinya
# (bukan 'skiprows=4' yang rapuh kalau jumlah baris judul berubah).
raw = pd.read_csv(FILE_LAHAN, header=None, dtype=str, keep_default_na=False,
                  encoding="utf-8-sig", skip_blank_lines=False)

POLA_KECAMATAN = r"^\s*\[(\d+)\]\s*(.+?)\s*$"
adalah_data = raw[0].str.match(POLA_KECAMATAN)
if not adalah_data.any():
    raise ValueError("Tidak ada baris berformat '[kode] Nama Kecamatan' di file lahan.")

idx_awal = int(np.argmax(adalah_data.values))
baris_judul = raw.iloc[:idx_awal]
print(f"Baris judul: {idx_awal} baris. Data kecamatan mulai di baris ke-{idx_awal + 1}.")

# Cari baris label kolom & baris tahun di antara baris judul.
label_kolom, tahun_kolom = None, None
for _, r in baris_judul.iterrows():
    sel = [str(x).strip() for x in r.iloc[1:5]]
    if any("rice fields" in s.lower() for s in sel):
        label_kolom = [s.lower() for s in sel]
    if all(re.fullmatch(r"\d{4}", s) for s in sel):
        tahun_kolom = sel

if label_kolom is None:
    raise ValueError("Baris label kolom (Rice fields / Not Rice fields / ...) tidak ditemukan.")

# Pastikan urutan kolom sesuai asumsi sebelum diberi nama.
urutan_ok = (
    "rice fields" in label_kolom[0] and "not" not in label_kolom[0]
    and "not rice" in label_kolom[1]
    and "not agri" in label_kolom[2]
    and "number" in label_kolom[3]
)
if not urutan_ok:
    raise ValueError(f"Urutan kolom lahan tidak sesuai asumsi: {label_kolom}")

if not any("hectare" in str(x).lower() for x in baris_judul.values.ravel()):
    print("⚠️  Satuan 'Hectare' tidak terlihat di baris judul. Pastikan satuannya hektar.")

if tahun_kolom is None:
    TAHUN_LAHAN = None
    print("⚠️  Baris tahun tidak ditemukan di judul.")
elif len(set(tahun_kolom)) == 1:
    TAHUN_LAHAN = int(tahun_kolom[0])
    print(f"✓ Tahun data lahan: {TAHUN_LAHAN}")
else:
    raise ValueError(f"Tahun berbeda antar kolom: {tahun_kolom}")

print("✓ Urutan kolom: sawah | pertanian bukan sawah | bukan pertanian | total")

Baris judul: 4 baris. Data kecamatan mulai di baris ke-5.
✓ Tahun data lahan: 2023
✓ Urutan kolom: sawah | pertanian bukan sawah | bukan pertanian | total


In [9]:
# 9. LUAS LAHAN PER KECAMATAN — BERSIHKAN

bagian_data = raw.iloc[idx_awal:]
mask_data = adalah_data.iloc[idx_awal:]

# Baris setelah tabel (total, catatan kaki, dll.) yang bukan kecamatan: laporkan, lalu buang.
sisa = bagian_data.loc[~mask_data]
sisa = sisa[sisa.apply(lambda r: any(str(x).strip() for x in r), axis=1)]
if len(sisa):
    print(f"Baris non-kecamatan yang dibuang ({len(sisa)}):")
    display(sisa)

baris_kec = bagian_data.loc[mask_data]
kode_nama = baris_kec[0].str.extract(POLA_KECAMATAN)

df_lahan = pd.DataFrame({
    "kode_kecamatan": kode_nama[0].values,
    "kecamatan": [KOREKSI_KECAMATAN.get(rapikan_nama(n), rapikan_nama(n)) for n in kode_nama[1]],
    "tahun": TAHUN_LAHAN,
})
for i, nama in enumerate(["luas_sawah_ha", "luas_pertanian_bukan_sawah_ha",
                          "luas_bukan_pertanian_ha", "luas_total_ha"], start=1):
    df_lahan[nama] = baris_kec[i].apply(ke_angka).values

# Luas hektar utuh -> simpan sebagai bilangan bulat kalau memang semuanya bulat.
kolom_luas = [c for c in df_lahan.columns if c.startswith("luas_")]
for c in kolom_luas:
    if df_lahan[c].notna().all() and (df_lahan[c] == df_lahan[c].round()).all():
        df_lahan[c] = df_lahan[c].astype(int)

df_lahan = df_lahan.reset_index(drop=True)
display(df_lahan)

,kode_kecamatan,kecamatan,tahun,luas_sawah_ha,luas_pertanian_bukan_sawah_ha,luas_bukan_pertanian_ha,luas_total_ha
0,010,Mojo,2023,926,7371,1866,10163
1,020,Semen,2023,1391,5579,901,7871
2,030,Ngadiluwih,2023,1160,984,1814,3958
3,040,Kras,2023,2191,513,1844,4548
4,050,Ringinrejo,2023,1590,555,2083,4228
5,060,Kandat,2023,1984,1048,1841,4873
6,070,Wates,2023,3370,241,2902,6513
7,080,Ngancar,2023,1508,4847,1726,8081
8,090,Plosoklaten,2023,2153,2329,2276,6758
9,100,Gurah,2023,2272,382,2101,4755


In [10]:
# 10. LUAS LAHAN — VALIDASI

# a) Kelengkapan & nama kecamatan
assert len(df_lahan) == len(MASTER_KECAMATAN), f"Jumlah kecamatan = {len(df_lahan)}, bukan {len(MASTER_KECAMATAN)}"
assert not df_lahan["kecamatan"].duplicated().any(), "Nama kecamatan ganda."
assert not df_lahan["kode_kecamatan"].duplicated().any(), "Kode kecamatan ganda."
selisih_nama = set(df_lahan["kecamatan"]) ^ set(MASTER_KECAMATAN)
assert not selisih_nama, f"Nama kecamatan tidak sama dengan master: {selisih_nama}"

# b) Nilai: tidak kosong, tidak negatif
assert df_lahan[kolom_luas].notna().all().all(), "Ada nilai luas yang kosong."
assert (df_lahan[kolom_luas] >= 0).all().all(), "Ada nilai luas negatif."

# c) Total = sawah + pertanian bukan sawah + bukan pertanian
komponen = (df_lahan["luas_sawah_ha"] + df_lahan["luas_pertanian_bukan_sawah_ha"]
            + df_lahan["luas_bukan_pertanian_ha"])
tidak_cocok = df_lahan.loc[komponen != df_lahan["luas_total_ha"], ["kecamatan", "luas_total_ha"]]
if len(tidak_cocok):
    print("⚠️  Total tidak sama dengan jumlah komponen di:")
    display(tidak_cocok.assign(jumlah_komponen=komponen[tidak_cocok.index]))
else:
    print("✓ Total lahan = sawah + pertanian bukan sawah + bukan pertanian, di semua kecamatan.")

print(f"✓ {len(df_lahan)} kecamatan, nama sama dengan master, kode unik.")
print(f"  Total luas sawah Kabupaten: {df_lahan['luas_sawah_ha'].sum():,} ha")

nol = df_lahan.loc[df_lahan[kolom_luas].eq(0).any(axis=1), ["kecamatan"] + kolom_luas]
if len(nol):
    print("ℹ️  Kecamatan dengan nilai 0 (nilai asli di sumber, bukan data hilang):")
    display(nol)

✓ Total lahan = sawah + pertanian bukan sawah + bukan pertanian, di semua kecamatan.
✓ 26 kecamatan, nama sama dengan master, kode unik.
  Total luas sawah Kabupaten: 44,066 ha
ℹ️  Kecamatan dengan nilai 0 (nilai asli di sumber, bukan data hilang):


,kecamatan,luas_sawah_ha,luas_pertanian_bukan_sawah_ha,luas_bukan_pertanian_ha,luas_total_ha
17,Purwoasri,2778,0,1133,3911


In [11]:
# 11. CEK SILANG: TANAMAN PANGAN vs LUAS LAHAN

total_sawah = df_lahan["luas_sawah_ha"].sum()
padi = df_pangan.loc[df_pangan["komoditas"] == "Padi", "luas_panen_ha"]
if len(padi):
    rasio = padi.iloc[0] / total_sawah
    print(f"Luas panen padi {padi.iloc[0]:,.2f} ha / luas sawah {total_sawah:,} ha = {rasio:.2f}")
    if rasio < 1:
        print("⚠️  Rasio < 1: luas panen padi lebih kecil dari luas sawah. Umumnya padi dipanen lebih dari")
        print("    sekali setahun sehingga rasionya di atas 1. Pastikan data panen & data lahan sama-sama")
        print("    untuk tahun yang sama dan periode yang sama (setahun penuh atau hanya satu musim).")

Luas panen padi 34,198.23 ha / luas sawah 44,066 ha = 0.78
⚠️  Rasio < 1: luas panen padi lebih kecil dari luas sawah. Umumnya padi dipanen lebih dari
    sekali setahun sehingga rasionya di atas 1. Pastikan data panen & data lahan sama-sama
    untuk tahun yang sama dan periode yang sama (setahun penuh atau hanya satu musim).


In [12]:
# 12. SIMPAN

path_pangan = os.path.join(OUT_DIR, "tanaman_pangan_kabupaten.csv")
path_lahan = os.path.join(OUT_DIR_LAHAN, "luas_lahan_per_kecamatan.csv")

df_pangan.to_csv(path_pangan, index=False, encoding="utf-8-sig")
df_lahan.to_csv(path_lahan, index=False, encoding="utf-8-sig")

# Baca ulang untuk memastikan isi file sama dengan yang di memori (kode tetap string, '010' bukan 10).
cek_pangan = pd.read_csv(path_pangan, encoding="utf-8-sig")
cek_lahan = pd.read_csv(path_lahan, encoding="utf-8-sig", dtype={"kode_kecamatan": str})
assert len(cek_pangan) == len(df_pangan) and len(cek_lahan) == len(df_lahan)
assert (cek_lahan["kode_kecamatan"] == df_lahan["kode_kecamatan"]).all()

print("✓ Tersimpan:")
print("  ", path_pangan)
print("  ", path_lahan)
display(cek_pangan)
display(cek_lahan.head())

✓ Tersimpan:
   d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\tanaman_pangan\tanaman_pangan_kabupaten.csv
   d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\tanaman_pangan\luas_lahan_per_kecamatan.csv


,komoditas,luas_panen_ha,produksi_ton,produktivitas_kuintal_ha
0,Padi,34198.23,198324.320000,57.992569
1,Kedelai,1.90,2.503035,13.173868
2,Kacang Tanah,476.50,843.696458,17.706117
3,Ubi Jalar,538.50,15825.831700,293.887311
4,Ketela Pohon,652.80,18597.024533,284.880891


,kode_kecamatan,kecamatan,tahun,luas_sawah_ha,luas_pertanian_bukan_sawah_ha,luas_bukan_pertanian_ha,luas_total_ha
0,010,Mojo,2023,926,7371,1866,10163
1,020,Semen,2023,1391,5579,901,7871
2,030,Ngadiluwih,2023,1160,984,1814,3958
3,040,Kras,2023,2191,513,1844,4548
4,050,Ringinrejo,2023,1590,555,2083,4228


## Yang perlu kamu cek manual
1. **Tahun data tanaman pangan tidak tercatat** di file sumbernya, sedangkan data lahan tahun 2023.
   Pastikan keduanya satu tahun sebelum dipakai bersama.
2. **Jagung dan Kacang Hijau belum ada** (keduanya disebut di proposal). "Ketela Pohon" dibiarkan sesuai
   sumber; penyeragaman ke "Ubi Kayu" sebaiknya lewat tabel master komoditas.
3. **Luas panen Kedelai hanya 1,9 ha** untuk seluruh kabupaten. Cek ke tabel BPS aslinya.
4. **Rasio luas panen padi terhadap luas sawah di bawah 1** (lihat cell 11). Kemungkinan beda tahun/periode.
5. Nilai `0` di lahan (Purwoasri, lahan pertanian bukan sawah) adalah nilai asli sumber, bukan data hilang.